# Planar Predictive Coding — minimal, in your backpropagation style

**Architecture:** `X (2,m) → H1 (6,m) → H2 (4,m) → p (1,m)`.

We keep your original seven-part idea. Unlike backpropagation, **training** has an *inner loop* that adjusts `H1` and `H2` with weights fixed, followed by an *outer update* that changes weights with `H1` and `H2` fixed. The output is supervised with **binary cross-entropy**; hidden-layer prediction errors are squared errors.

**Run from the folder containing `planar_utils.py`** (the same utility your original notebook imports). This notebook leaves your uploaded files unchanged.

**Our steps**
1. Load data
2. Define layers
3. Initialize weights
4. Forward propagation
5. Compute prediction errors and energy; infer hidden states
6. Compute local weight gradients and update weights
7. Integrate training, then calculate accuracy

## 1. Import libraries and dataset — **unchanged**

In [25]:
import numpy as np
from planar_utils import load_planar_dataset, sigmoid

X, Y = load_planar_dataset()
print("X:", X.shape, "Y:", Y.shape)  # (2, 400), (1, 400)

X: (2, 400) Y: (1, 400)


## 2. Define layers — **unchanged**

In [26]:
def layers(X, Y):
    n_x = X.shape[0]
    h1 = 6
    h2 = 4
    n_y = Y.shape[0]
    return (n_x, h1, h2, n_y)

## 3. Initialize weights and biases — **unchanged**

We keep the same weight shapes and zero biases. `W1`/`W2` use your original small random initialization, while `W3` uses a larger random scale so the output error gives a useful signal to hidden activities. We set the seed once for reproducible experiments. Each new batch will have *new inferred activities*, but **not** newly initialized weights.

In [27]:
def initialize(n_x, h1, h2, n_y):
    W1 = np.random.rand(h1, n_x) * 0.01
    b1 = np.zeros((h1, 1))
    W2 = np.random.rand(h2, h1) * 0.01
    b2 = np.zeros((h2, 1))
    W3 = np.random.rand(n_y, h2)  # slightly larger output weights help signal reach hidden layers
    b3 = np.zeros((n_y, 1))

    return {"W1": W1, "W2": W2, "W3": W3,
            "b1": b1, "b2": b2, "b3": b3}

## 4. Forward propagation — **same equations as backprop**

In PC this also gives a sensible **initial guess** for adjustable hidden activities. During training, `H1 = A1.copy()` and `H2 = A2.copy()`; afterward `H1` and `H2` can move independently of the forward activations. We do **not** optimize a separate `H3`: the target `Y` is fixed and the output BCE supplies its error signal.

In [28]:
def forwared_propagation(X, parameters):
    W1, W2, W3 = parameters["W1"], parameters["W2"], parameters["W3"]
    b1, b2, b3 = parameters["b1"], parameters["b2"], parameters["b3"]

    Z1 = W1 @ X + b1
    A1 = np.tanh(Z1)
    Z2 = W2 @ A1 + b2
    A2 = np.tanh(Z2)
    Z3 = W3 @ A2 + b3
    A3 = sigmoid(Z3)

    return {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2,
            "Z3": Z3, "A3": A3}

## 5a. Calculate prediction errors — **new**

For each layer, **error = current activity − predicted activity**:

$$
\mu_1=\tanh(W_1X+b_1),\quad e_1=H_1-\mu_1;
\qquad \mu_2=\tanh(W_2H_1+b_2),\quad e_2=H_2-\mu_2.
$$

For the output, $p=\sigma(W_3H_2+b_3)$ and $dZ_3=p-Y$ is the **BCE derivative with respect to its logit**, not another hidden squared-error residual. Notice `mu2` depends on **current `H1`**, not the old forward `A1`.

In [29]:
def compute_errors(X, Y, parameters, H1, H2):
    mu1 = np.tanh(parameters["W1"] @ X + parameters["b1"])
    mu2 = np.tanh(parameters["W2"] @ H1 + parameters["b2"])
    Z3 = parameters["W3"] @ H2 + parameters["b3"]
    p = sigmoid(Z3)

    e1 = H1 - mu1
    e2 = H2 - mu2
    dZ3 = p - Y

    return {"mu1": mu1, "mu2": mu2, "Z3": Z3, "p": p,
            "e1": e1, "e2": e2, "dZ3": dZ3}

## 5b. Calculate energy — **one scalar assembled from the errors**

$$
E=\frac1m\left[\frac12\sum e_1^2+\frac12\sum e_2^2+
\sum_j\underbrace{\big(\log(1+e^{Z_{3,j}})-Y_jZ_{3,j}\big)}_{\mathrm{BCE\ from\ logits}}\right].
$$

`compute_cost(Y, cache)` measures ordinary forward BCE, exactly as in your backprop workflow. `compute_energy` is mainly useful for understanding and checking whether inference is improving the objective. `compute_errors` gives the *local signals* used in the derivatives. Because we start `H1, H2` from the forward pass, both hidden errors initially equal zero.

In [30]:
def compute_cost(Y, cache):
    # Ordinary forward BCE: use for evaluating predictions.
    return np.mean(np.logaddexp(0, cache["Z3"]) - Y * cache["Z3"])


def compute_energy(Y, errors):
    # PC energy: use to understand hidden-state inference and weight gradients.
    hidden_loss = 0.5 * np.sum(errors["e1"]**2) + 0.5 * np.sum(errors["e2"]**2)
    output_loss = np.sum(np.logaddexp(0, errors["Z3"]) - Y * errors["Z3"])
    m = Y.shape[1]
    return (hidden_loss + output_loss) / m

## 5c. Infer hidden states — **the main difference from backprop**

For one sample, differentiating the energy with respect to each **activity** yields

$$
\boxed{g_{H_2}= e_2+W_3^T(p-Y)}\qquad
\boxed{g_{H_1}=e_1-W_2^T\big(e_2\odot(1-\mu_2^2)\big)}.
$$

These are derivatives of the **per-sample** energy; they omit the harmless `1/m` batch average so `activity_rate` works independently of batch size. For both states: `H = H - activity_rate * gradient`. Keep the weights fixed during these steps. Recalculate predictions and errors after every update. We calculate *both* gradients from the old states before moving either one.

At forward initialization, `e1 = e2 = 0`, so the **first** step changes only `H2`; on later steps that change produces an `e2` signal that can move `H1`. This is how the target's influence travels through the network over iterations.

In [31]:
def inference(X, Y, parameters, cache, steps=20, activity_rate=0.1):
    H1 = cache["A1"].copy()  # (6, m): per-sample state
    H2 = cache["A2"].copy()  # (4, m): per-sample state

    for i in range(steps):
        errors = compute_errors(X, Y, parameters, H1, H2)

        # d(energy of one sample)/d(hidden activities)
        dH2 = errors["e2"] + parameters["W3"].T @ errors["dZ3"]
        dH1 = errors["e1"] - parameters["W2"].T @ (errors["e2"] * (1 - errors["mu2"]**2))

        # Update the ACTIVITIES, not the weights
        H1 = H1 - activity_rate * dH1
        H2 = H2 - activity_rate * dH2

    errors = compute_errors(X, Y, parameters, H1, H2)
    return H1, H2, errors

## 6a. Calculate local weight gradients — **new way to get `dW` and `db`**

Now hold the inferred `H1` and `H2` fixed. Each layer adjusts its **weights** to better predict its current target activity:

$$
\begin{aligned}
 dZ_1&=-e_1\odot(1-\mu_1^2), &dW_1&=dZ_1X^T/m,\\
 dZ_2&=-e_2\odot(1-\mu_2^2), &dW_2&=dZ_2H_1^T/m,\\
 dZ_3&=p-Y, &dW_3&=dZ_3H_2^T/m.
\end{aligned}
$$

The biases use `np.sum(dZi, axis=1, keepdims=True) / m`. The minus signs at layers 1 and 2 follow from `e = activity - prediction`. Matrix multiplication uses `@` (equivalent to `np.dot` for these two-dimensional arrays). We divide by `m` because the **weight objective is averaged across samples**.

In [32]:
def parameter_gradients(X, Y, parameters, H1, H2, errors):
    m = X.shape[1]

    dZ1 = -errors["e1"] * (1 - errors["mu1"]**2)
    dZ2 = -errors["e2"] * (1 - errors["mu2"]**2)
    dZ3 = errors["dZ3"]

    dW1 = (dZ1 @ X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m
    dW2 = (dZ2 @ H1.T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m
    dW3 = (dZ3 @ H2.T) / m
    db3 = np.sum(dZ3, axis=1, keepdims=True) / m

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2,
            "dW3": dW3, "db3": db3}

## 6b. Update weights — **same subtraction as backpropagation**

In [33]:
def weight_update(parameters, grads, weight_rate=0.1):
    for name in ("W1", "b1", "W2", "b2", "W3", "b3"):
        parameters[name] = parameters[name] - weight_rate * grads["d" + name]
    return parameters

## 7. Integrate training — **one inner loop before one weight update**

- Start with the forward pass and initialize `H1`, `H2`.
- Run `inference(...)` for several steps, using `Y` but **without changing weights**.
- Calculate the weight gradients at the *final inferred state* and update weights **once**.
- Print **ordinary forward BCE and accuracy** every 1,000 weight updates. This measures how well the network predicts `Y` without seeing labels at test time; it is **not** the same as PC's relaxed training energy.

In [34]:
def predict(X, parameters):
    cache = forwared_propagation(X, parameters)
    return (cache["A3"] >= 0.5).astype(int)


def pc_model(X, Y, h1=6, h2=4, iterations=10000,
             steps=20, activity_rate=0.1, weight_rate=0.1):
    n_x, _, _, n_y = layers(X, Y)
    parameters = initialize(n_x, h1, h2, n_y)
    history = []

    for i in range(iterations + 1):
        # Evaluate with the ordinary forward pass (same at test time)
        cache = forwared_propagation(X, parameters)
        if i % 1000 == 0 or i == iterations:
            forward_bce = compute_cost(Y, cache)
            accuracy = np.mean((cache["A3"] >= 0.5) == Y)
            history.append((i, float(forward_bce), float(accuracy)))
            print(f"Iteration {i:5d}: BCE = {forward_bce:.5f}, accuracy = {accuracy:.2%}")

        if i == iterations:
            break

        # Train: first infer H1/H2, THEN learn W/b from local errors
        H1, H2, errors = inference(X, Y, parameters, cache, steps, activity_rate)
        grads = parameter_gradients(X, Y, parameters, H1, H2, errors)
        parameters = weight_update(parameters, grads, weight_rate)

    return parameters, history

## Run the model

For learning, `W1` and `W2` keep your small initialization; `W3` is initialized on a larger scale to help the output signal reach the hidden activities. Once you understand inference, try changing the activity rate, inference steps and initialization. This is a training-set demonstration, not a validation/test accuracy estimate.

In [35]:
np.random.seed(3)
parameters, history = pc_model(X, Y, h1=6, h2=4, iterations=10000,
                               steps=20, activity_rate=0.1, weight_rate=0.1)

Iteration     0: BCE = 0.69320, accuracy = 40.00%


/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:7: RuntimeWarning: divide by zero encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:7: RuntimeWarning: overflow encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:7: RuntimeWarning: invalid value encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:9: RuntimeWarning: divide by zero encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:9: RuntimeWarning: overflow encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:9: RuntimeWarning: invalid value encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/467788396.py:3: RuntimeWarning: 

Iteration  1000: BCE = 0.69547, accuracy = 47.00%
Iteration  2000: BCE = 0.70236, accuracy = 47.00%
Iteration  3000: BCE = 0.70212, accuracy = 46.50%
Iteration  4000: BCE = 0.67444, accuracy = 49.50%
Iteration  5000: BCE = 0.58152, accuracy = 75.50%
Iteration  6000: BCE = 0.47452, accuracy = 81.00%
Iteration  7000: BCE = 0.45023, accuracy = 83.25%
Iteration  8000: BCE = 0.43956, accuracy = 84.00%
Iteration  9000: BCE = 0.44086, accuracy = 84.25%
Iteration 10000: BCE = 0.45245, accuracy = 84.50%


## Tiny sanity check: does hidden inference lower energy?

This is optional but useful when debugging. We run inference once with the current weights and compare energy before and after; reducing energy is expected for a sufficiently small activity learning rate, not guaranteed for arbitrary rates.

In [36]:
cache = forwared_propagation(X, parameters)
start_H1, start_H2 = cache["A1"].copy(), cache["A2"].copy()
start_errors = compute_errors(X, Y, parameters, start_H1, start_H2)
H1, H2, end_errors = inference(X, Y, parameters, cache, steps=20, activity_rate=0.1)
print("Initial hidden residuals:", np.max(np.abs(start_errors["e1"])), np.max(np.abs(start_errors["e2"])))
print("Energy before:", compute_energy(Y, start_errors))
print("Energy after: ", compute_energy(Y, end_errors))

Initial hidden residuals: 0.0 0.0
Energy before: 0.4524509162016224
Energy after:  0.010454007740517045


/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:7: RuntimeWarning: divide by zero encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:7: RuntimeWarning: overflow encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:7: RuntimeWarning: invalid value encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:9: RuntimeWarning: divide by zero encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:9: RuntimeWarning: overflow encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/364709847.py:9: RuntimeWarning: invalid value encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_5113/467788396.py:3: RuntimeWarning: 